# SQLibrarian: baseline of 4 models (Kaggle)

Asks each candidate model the 50 dev questions with no training, and saves the SQL it writes.
Scoring happens afterwards **on your laptop**, where `records_db` runs (`evaluation/score.py`).
Cost: free (Kaggle GPU quota, about 20–40 minutes for all four runs).

**Before you run it:**
1. Push the `p0-setup` branch to GitHub (the notebook clones it).
2. Settings (right panel): Accelerator **GPU T4 x2**, Internet **on**.
3. Optional: Add-ons, Secrets, `HF_TOKEN` (any read token; avoids download rate limits).

| Section | What it does | Done when |
|---|---|---|
| A | Settings: branch and the four runs | it prints them |
| B | Clone the repo and install it | "all imports OK" |
| C | Check the GPU | `nvidia-smi` lists a Tesla T4 |
| D | Run the four baselines, one process each | a summary line per run |
| E | Pack the predictions for download | `baseline_predictions.zip` in Output |

## A. Settings

In [ ]:
REPO_URL = "https://github.com/OmarAmir2001/sqlibrarian.git"
BRANCH = "p0-setup"
WORK = "/kaggle/working"
REPO_DIR = f"{WORK}/sqlibrarian-repo"  # not "sqlibrarian": that name would hide the package

# (model, few-shot?) — the four runs from the training guide, step 4
RUNS = [
    ("Qwen/Qwen2.5-Coder-1.5B-Instruct", False),
    ("Qwen/Qwen2.5-Coder-3B-Instruct", False),
    ("Qwen/Qwen3.5-2B", False),
    ("Qwen/Qwen2.5-Coder-1.5B-Instruct", True),  # same 1.5B, with 3 example pairs
]
for model, few_shot in RUNS:
    print(model, "+ 3 examples" if few_shot else "")

## B. Get the code
Same install as the data notebook: no dependencies, so Kaggle's GPU-matched PyTorch stays.

In [ ]:
import os
import subprocess
import sys

try:
    from kaggle_secrets import UserSecretsClient

    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("using HF_TOKEN")
except Exception:
    print("no HF_TOKEN secret: downloading anonymously (fine for public models)")

# Step out of the old clone first, so re-running this cell is safe.
os.chdir(WORK)
subprocess.run(["rm", "-rf", REPO_DIR], check=True)
subprocess.run(
    ["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True
)
%cd $REPO_DIR
!git log --oneline -1

In [ ]:
!pip install -q uv
!uv pip install --system -q --no-deps -e .
!uv pip install --system -q sqlglot

In [ ]:
# Checked in a NEW Python process, like every run below.
check = "import sqlibrarian.evaluation.generate, torch, transformers; print('all imports OK, transformers', transformers.__version__)"
subprocess.run([sys.executable, "-c", check], check=True)

## C. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## D. Run the four baselines
Each run is its own Python process, so GPU memory is fully freed between models,
and one failing model (e.g. a model this Kaggle image's `transformers` is too old for)
doesn't stop the others. Each prints one line per question; about 1–3 s per question on a T4.

In [ ]:
failed = []
for model, few_shot in RUNS:
    cmd = [sys.executable, "-m", "sqlibrarian.evaluation.generate", "--model", model]
    cmd += ["--few-shot"] if few_shot else []
    print("\n===", " ".join(cmd[2:]))
    if subprocess.run(cmd).returncode != 0:
        failed.append(model + (" (few-shot)" if few_shot else ""))
print("\nfailed runs:", failed or "none")

If `Qwen/Qwen3.5-2B` failed with an unknown model type, this image's `transformers` is older
than the model. It is compared in the baseline only, so you can skip it and note that in the tracker,
or run `!pip install -q -U transformers` in a new cell, **restart the session**, and run sections B–D again.

## E. Pack the predictions for download
Then open the Output tab (right panel) and download the zip.

In [ ]:
import json
from pathlib import Path

pred_dir = Path("data/eval/predictions")
for meta in sorted(pred_dir.glob("*.meta.json")):
    m = json.loads(meta.read_text())
    print(
        f"{m['run']:38s} {m['mean_seconds']:5.2f} s/question  p95 {m['p95_seconds']:5.2f} s  peak GPU {m['peak_gpu_memory_gb']} GB"
    )
!cd data/eval && zip -q -r /kaggle/working/baseline_predictions.zip predictions
print("saved /kaggle/working/baseline_predictions.zip")